# IndoBERT Sentiment Fine-Tuning Run 2 — Weighted Loss


## Research Context
Run 1 baseline IndoBERT achieved accuracy `0.7416`, macro F1 `0.6927`, and weighted F1 `0.7365`. The main weakness was the `Neutral` class, with recall `0.4526` and F1 `0.5021`. Run 2 tests weighted cross entropy loss to improve Neutral recall and macro F1. This is an experiment, not a guaranteed final model.

## Experiment Comparison Setup
Run 2 writes to a separate Google Drive folder and must not overwrite the Run 1 baseline.

In [ ]:
RUN_NAME = "run_2_weighted_loss"
BASELINE_RUN_NAME = "run_1_baseline"


## Environment Setup
Run this notebook in Google Colab with GPU runtime enabled. Package installation is intended for Colab only.

In [ ]:
!pip -q install torch transformers datasets evaluate accelerate scikit-learn pandas matplotlib


## GPU Check
Stop early if CUDA is unavailable.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU runtime is required for Run 2 IndoBERT fine-tuning.")


## Google Drive Mount
The base path is configurable through `DRIVE_PROJECT_DIR`. Keep Run 2 outputs separate from Run 1 baseline outputs.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/SentiRank")
DATASET_DIR = DRIVE_PROJECT_DIR / "datasets" / "processed" / "indobert"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs" / "indobert" / RUN_NAME
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "saved_model"
BASELINE_OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs" / "indobert" / BASELINE_RUN_NAME

for output_path in [OUTPUT_DIR, FIGURE_DIR, MODEL_OUTPUT_DIR]:
    output_path.mkdir(parents=True, exist_ok=True)

print("Run 2 output:", OUTPUT_DIR)
print("Run 1 baseline:", BASELINE_OUTPUT_DIR)


## Dataset Paths
Expected input files in `DATASET_DIR`:
- `train.csv`
- `validation.csv`
- `test.csv`
- `label_mapping.json`

In [ ]:
TRAIN_FILE = DATASET_DIR / "train.csv"
VALIDATION_FILE = DATASET_DIR / "validation.csv"
TEST_FILE = DATASET_DIR / "test.csv"
LABEL_MAPPING_FILE = DATASET_DIR / "label_mapping.json"

for file_path in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE, LABEL_MAPPING_FILE]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing required input file: {file_path}")


## Dataset Loading and Validation
Required columns are `text_indobert`, `label_id`, and `final_sentiment`. The expected label mapping is `0 = Negative`, `1 = Neutral`, and `2 = Positive`.

In [ ]:
import json
import random

import numpy as np
import pandas as pd

REQUIRED_COLUMNS = {"text_indobert", "label_id", "final_sentiment"}
EXPECTED_LABEL_MAPPING = {"Negative": 0, "Neutral": 1, "Positive": 2}

def load_split(path: Path, split_name: str) -> pd.DataFrame:
    data = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS - set(data.columns)
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing_columns)}")
    empty_text_count = data["text_indobert"].fillna("").astype(str).str.strip().eq("").sum()
    if empty_text_count:
        raise ValueError(f"{split_name} contains {empty_text_count} empty text rows")
    return data

train_df = load_split(TRAIN_FILE, "train")
validation_df = load_split(VALIDATION_FILE, "validation")
test_df = load_split(TEST_FILE, "test")
label_mapping = json.loads(LABEL_MAPPING_FILE.read_text(encoding="utf-8"))

if label_mapping != EXPECTED_LABEL_MAPPING:
    raise ValueError(f"Unexpected label mapping: {label_mapping}")

id2label = {label_id: label for label, label_id in label_mapping.items()}
label2id = label_mapping

print("Dataset sizes:")
print({"train": len(train_df), "validation": len(validation_df), "test": len(test_df)})

for split_name, data in {"train": train_df, "validation": validation_df, "test": test_df}.items():
    print(f"\n{split_name} label distribution")
    print(data["label_id"].map(id2label).value_counts().sort_index())


## Class Weight Calculation
Compute balanced class weights from the training split. Run 1 train distribution was expected to be Negative `27780`, Neutral `12340`, Positive `28326`; therefore Neutral should receive the highest weight.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2])
train_labels = train_df["label_id"].astype(int).to_numpy()
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)
class_weights_payload = {
    "run_name": RUN_NAME,
    "method": "sklearn.utils.class_weight.compute_class_weight(class_weight='balanced')",
    "train_distribution": train_df["label_id"].map(id2label).value_counts().sort_index().to_dict(),
    "weights_by_label": {id2label[int(label_id)]: float(weight) for label_id, weight in zip(classes, weights)},
    "weights_by_id": {str(int(label_id)): float(weight) for label_id, weight in zip(classes, weights)},
}

(OUTPUT_DIR / "indobert_class_weights.json").write_text(
    json.dumps(class_weights_payload, indent=2, sort_keys=True), encoding="utf-8"
)
class_weights_payload


## Tokenization
Use `indobenchmark/indobert-base-p1`, `max_length = 128`, and `AutoTokenizer` to tokenize `text_indobert`.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(data: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(
        data[["text_indobert", "label_id"]].rename(columns={"label_id": "labels"}),
        preserve_index=False,
    )

def tokenize_batch(batch):
    return tokenizer(batch["text_indobert"], truncation=True, max_length=MAX_LENGTH)

train_dataset = to_hf_dataset(train_df).map(tokenize_batch, batched=True)
validation_dataset = to_hf_dataset(validation_df).map(tokenize_batch, batched=True)
test_dataset = to_hf_dataset(test_df).map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## Model Setup
Use `AutoModelForSequenceClassification` with three labels and explicit `id2label` / `label2id` mappings.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)


## Custom Weighted Trainer
Override `compute_loss` and use weighted cross entropy. Class weights are moved to the same device as model logits.

In [ ]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        if labels is None:
            raise ValueError("WeightedTrainer requires labels in inputs.")
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        loss_function = torch.nn.CrossEntropyLoss(
            weight=self.class_weights_tensor.to(logits.device)
        )
        loss = loss_function(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


## Training Configuration
Run 2 config: model `indobenchmark/indobert-base-p1`, max length `128`, batch size `16`, learning rate `2e-5`, epochs `3`, random state `42`, epoch evaluation/save, and best model by `f1_macro`. If GPU memory is limited, use batch size `8` with `gradient_accumulation_steps = 2`.

In [ ]:
import inspect

from transformers import EarlyStoppingCallback, TrainingArguments

BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 2e-5
EPOCHS = 3
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

training_argument_names = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
strategy_key = "eval_strategy" if "eval_strategy" in training_argument_names else "evaluation_strategy"
training_kwargs = {
    "output_dir": str(OUTPUT_DIR / "checkpoints"),
    strategy_key: "epoch",
    "save_strategy": "epoch",
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": EPOCHS,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "greater_is_better": True,
    "report_to": [],
    "seed": RANDOM_STATE,
}
if "logging_strategy" in training_argument_names:
    training_kwargs["logging_strategy"] = "epoch"
if "save_total_limit" in training_argument_names:
    training_kwargs["save_total_limit"] = 1

training_args = TrainingArguments(**training_kwargs)


## Metrics Function
Compute accuracy, macro precision, macro recall, macro F1, weighted precision, weighted recall, and weighted F1.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }


## Training
Use `WeightedTrainer`, save the best model, and print best checkpoint, best metric, and runtime.

In [ ]:
trainer_argument_names = set(inspect.signature(Trainer.__init__).parameters.keys())
trainer_tokenizer_kwargs = (
    {"processing_class": tokenizer}
    if "processing_class" in trainer_argument_names
    else {"tokenizer": tokenizer}
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights_tensor=class_weights_tensor,
    **trainer_tokenizer_kwargs,
)

train_result = trainer.train()
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)
print("Final training runtime:", train_result.metrics.get("train_runtime"))


## Save Training History
Save `trainer.state.log_history` to CSV and JSON for later analysis.

In [ ]:
history_df = pd.DataFrame(trainer.state.log_history)
preferred_history_columns = [
    "epoch",
    "loss",
    "eval_loss",
    "eval_accuracy",
    "eval_precision_macro",
    "eval_recall_macro",
    "eval_f1_macro",
    "eval_f1_weighted",
    "learning_rate",
]
ordered_history_columns = [column for column in preferred_history_columns if column in history_df.columns]
remaining_history_columns = [column for column in history_df.columns if column not in ordered_history_columns]
history_df = history_df[ordered_history_columns + remaining_history_columns]
history_df.to_csv(OUTPUT_DIR / "indobert_training_history.csv", index=False)
(OUTPUT_DIR / "indobert_training_history.json").write_text(
    history_df.to_json(orient="records", indent=2), encoding="utf-8"
)
history_df.tail()


## Test Evaluation
Evaluate the best model on the test set and save metrics, classification report, and confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_metrics = trainer.evaluate(test_dataset, metric_key_prefix="test")
prediction_output = trainer.predict(test_dataset)
test_logits = prediction_output.predictions
test_probabilities = torch.nn.functional.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_predictions = np.argmax(test_probabilities, axis=-1)
test_labels = prediction_output.label_ids

classification_report_dict = classification_report(
    test_labels,
    test_predictions,
    labels=[0, 1, 2],
    target_names=[id2label[0], id2label[1], id2label[2]],
    output_dict=True,
    zero_division=0,
)
confusion_matrix_data = confusion_matrix(test_labels, test_predictions, labels=[0, 1, 2])

training_config = {
    "run_name": RUN_NAME,
    "baseline_run_name": BASELINE_RUN_NAME,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "random_state": RANDOM_STATE,
    "loss": "weighted_cross_entropy",
    "class_weights": class_weights_payload,
}
metrics_payload = {
    "training_config": training_config,
    "train_metrics": train_result.metrics,
    "test_metrics": test_metrics,
}
(OUTPUT_DIR / "indobert_training_metrics.json").write_text(
    json.dumps(metrics_payload, indent=2, sort_keys=True), encoding="utf-8"
)
(OUTPUT_DIR / "indobert_classification_report.json").write_text(
    json.dumps(classification_report_dict, indent=2, sort_keys=True), encoding="utf-8"
)
pd.DataFrame(
    confusion_matrix_data,
    index=[id2label[0], id2label[1], id2label[2]],
    columns=[id2label[0], id2label[1], id2label[2]],
).to_csv(OUTPUT_DIR / "indobert_confusion_matrix.csv")

test_metrics


## Prediction Probabilities
Save test predictions with probabilities for each class and confidence.

In [ ]:
trace_columns = [
    "external_id",
    "rating",
    "content",
    "text_indobert",
    "initial_sentiment",
    "final_sentiment",
]
available_trace_columns = [column for column in trace_columns if column in test_df.columns]
predictions_df = test_df[available_trace_columns].copy()
predictions_df["true_label_id"] = test_labels
predictions_df["predicted_label_id"] = test_predictions
predictions_df["true_label"] = [id2label[int(label_id)] for label_id in test_labels]
predictions_df["predicted_label"] = [id2label[int(label_id)] for label_id in test_predictions]
predictions_df["prob_negative"] = test_probabilities[:, 0]
predictions_df["prob_neutral"] = test_probabilities[:, 1]
predictions_df["prob_positive"] = test_probabilities[:, 2]
predictions_df["confidence"] = test_probabilities.max(axis=1)
predictions_df.to_csv(OUTPUT_DIR / "indobert_test_predictions.csv", index=False)
predictions_df.head()


## Error Analysis
Save only rows where the true label differs from the predicted label.

In [ ]:
error_analysis_df = predictions_df[predictions_df["true_label"] != predictions_df["predicted_label"]].copy()
error_columns = [
    column
    for column in [
        "external_id",
        "rating",
        "content",
        "text_indobert",
        "initial_sentiment",
        "final_sentiment",
        "true_label",
        "predicted_label",
        "prob_negative",
        "prob_neutral",
        "prob_positive",
        "confidence",
    ]
    if column in error_analysis_df.columns
]
error_analysis_df[error_columns].to_csv(OUTPUT_DIR / "indobert_error_analysis.csv", index=False)
print("Error rows:", len(error_analysis_df))
error_analysis_df[error_columns].head()


## Figures
Save confusion matrix, evaluation metrics, training loss, and per-class F1 figures using matplotlib only.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.imshow(confusion_matrix_data, cmap="Blues")
plt.title("IndoBERT Run 2 Confusion Matrix")
plt.xticks([0, 1, 2], [id2label[0], id2label[1], id2label[2]])
plt.yticks([0, 1, 2], [id2label[0], id2label[1], id2label[2]])
plt.xlabel("Predicted")
plt.ylabel("True")
for row_index in range(confusion_matrix_data.shape[0]):
    for column_index in range(confusion_matrix_data.shape[1]):
        plt.text(column_index, row_index, confusion_matrix_data[row_index, column_index], ha="center", va="center")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "indobert_confusion_matrix.png", dpi=160)
plt.show()

metric_names = ["test_accuracy", "test_precision_macro", "test_recall_macro", "test_f1_macro", "test_f1_weighted"]
metric_values = [float(test_metrics.get(name, 0.0)) for name in metric_names]
plt.figure(figsize=(9, 5))
plt.bar(metric_names, metric_values, color="#2f6f9f")
plt.title("IndoBERT Run 2 Test Metrics")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "indobert_eval_metrics.png", dpi=160)
plt.show()

loss_points = [(entry.get("epoch"), entry.get("loss")) for entry in trainer.state.log_history if "loss" in entry]
eval_loss_points = [(entry.get("epoch"), entry.get("eval_loss")) for entry in trainer.state.log_history if "eval_loss" in entry]
plt.figure(figsize=(7, 4))
if loss_points:
    epochs, losses = zip(*loss_points)
    plt.plot(epochs, losses, marker="o", label="Training loss")
if eval_loss_points:
    epochs, losses = zip(*eval_loss_points)
    plt.plot(epochs, losses, marker="o", label="Validation loss")
plt.title("IndoBERT Run 2 Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "indobert_training_loss.png", dpi=160)
plt.show()

class_names = [id2label[0], id2label[1], id2label[2]]
class_f1_scores = [classification_report_dict[class_name]["f1-score"] for class_name in class_names]
plt.figure(figsize=(7, 4))
plt.bar(class_names, class_f1_scores, color="#2f6f9f")
plt.title("IndoBERT Run 2 Per-Class F1")
plt.ylabel("F1-score")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "indobert_class_f1_score.png", dpi=160)
plt.show()


## Baseline Comparison Section
If Run 1 baseline metrics and classification report are available in Google Drive, compare Run 1 and Run 2. If they are missing, skip comparison safely.

In [ ]:
baseline_metrics_file = BASELINE_OUTPUT_DIR / "indobert_training_metrics.json"
baseline_report_file = BASELINE_OUTPUT_DIR / "indobert_classification_report.json"

def extract_test_metric(metrics: dict, metric_name: str):
    test_metric_name = f"test_{metric_name}"
    return metrics.get("test_metrics", {}).get(test_metric_name, metrics.get("test_metrics", {}).get(metric_name))

if baseline_metrics_file.is_file() and baseline_report_file.is_file():
    baseline_metrics = json.loads(baseline_metrics_file.read_text(encoding="utf-8"))
    baseline_report = json.loads(baseline_report_file.read_text(encoding="utf-8"))
    comparison_rows = [
        {
            "run_name": BASELINE_RUN_NAME,
            "accuracy": extract_test_metric(baseline_metrics, "accuracy"),
            "macro_f1": extract_test_metric(baseline_metrics, "f1_macro"),
            "weighted_f1": extract_test_metric(baseline_metrics, "f1_weighted"),
            "neutral_precision": baseline_report["Neutral"]["precision"],
            "neutral_recall": baseline_report["Neutral"]["recall"],
            "neutral_f1": baseline_report["Neutral"]["f1-score"],
        },
        {
            "run_name": RUN_NAME,
            "accuracy": test_metrics.get("test_accuracy"),
            "macro_f1": test_metrics.get("test_f1_macro"),
            "weighted_f1": test_metrics.get("test_f1_weighted"),
            "neutral_precision": classification_report_dict["Neutral"]["precision"],
            "neutral_recall": classification_report_dict["Neutral"]["recall"],
            "neutral_f1": classification_report_dict["Neutral"]["f1-score"],
        },
    ]
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(OUTPUT_DIR / "indobert_run_comparison.csv", index=False)
    (OUTPUT_DIR / "indobert_run_comparison.json").write_text(
        comparison_df.to_json(orient="records", indent=2), encoding="utf-8"
    )
    plot_metrics = ["accuracy", "macro_f1", "weighted_f1", "neutral_recall", "neutral_f1"]
    x_positions = np.arange(len(plot_metrics))
    width = 0.35
    plt.figure(figsize=(10, 5))
    plt.bar(x_positions - width / 2, comparison_df.loc[0, plot_metrics], width=width, label=BASELINE_RUN_NAME)
    plt.bar(x_positions + width / 2, comparison_df.loc[1, plot_metrics], width=width, label=RUN_NAME)
    plt.title("IndoBERT Run 1 vs Run 2")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.xticks(x_positions, plot_metrics, rotation=25, ha="right")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "indobert_run_comparison.png", dpi=160)
    plt.show()
    comparison_df
else:
    print("Run 1 baseline files were not found in Google Drive; baseline comparison skipped.")


## Interpretation Section
After training, document whether weighted loss improved Neutral recall, whether macro F1 improved, whether Negative or Positive performance dropped, whether Run 2 should become the candidate final model, and what limitations remain.

## Local Repository Sync Note
Copy small Run 2 metrics to `datasets/outputs/eda/03_indobert/run_2_weighted_loss/` and figures to `docs/figures/03_indobert/run_2_weighted_loss/`. Do not commit `saved_model/`, checkpoints, `model.safetensors`, `pytorch_model.bin`, or large prediction files if they are too large for the repository policy.

## Next Step
Compare Run 1 vs Run 2. If Run 2 improves macro F1 and Neutral recall without unacceptable degradation in Negative or Positive performance, use it as the candidate final IndoBERT model. If not, consider Run 3 with a lower learning rate.